# 04 · Urban heat and greening scenarios

**Question:** How does a raster intervention affect population-weighted exposure?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
All city geometry, residents, sensor observations and parameters are **synthetic educational fixtures**.
The longitude/latitude anchor is near Gaborone, but these are not mapped Gaborone assets or a validated city twin.

**Astra experiment:** Ask Astra to explain the map and distinguish area-average from population-weighted changes; score the numerical claims.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Synthetic raster, not satellite temperature
The heat field has degree-Celsius labels for teaching, but comes from an invented equation.
Nearest-pixel sampling is explicit; actual raster work needs CRS, nodata, acquisition time and resolution checks.


In [ ]:
c=city(); pop=allocate(c['capacity'],1200)
xx,yy,green,heat,elevation=raster()
cooling_c=2.5; radius_m=450.; center=(1400,700)
effect=cooling_c*np.exp(-((xx-center[0])**2+(yy-center[1])**2)/(2*radius_m**2))
after=heat-effect
ix=np.clip(np.rint(c['xy'][:,0]/2000*(heat.shape[1]-1)).astype(int),0,heat.shape[1]-1)
iy=np.clip(np.rint(c['xy'][:,1]/2000*(heat.shape[0]-1)).astype(int),0,heat.shape[0]-1)
before_mean=float(np.average(heat[iy,ix],weights=pop)); after_mean=float(np.average(after[iy,ix],weights=pop))
assert np.isfinite(after).all() and (after<=heat).all()
print('Population-weighted synthetic temperature:',round(before_mean,2),'->',round(after_mean,2))


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,field,title in zip(axes,[heat,after,effect],['Baseline','Greening scenario','Cooling difference']):
    m=ax.imshow(field,origin='lower',extent=[0,2000,0,2000],cmap='YlOrRd',vmin=heat.min() if title!='Cooling difference' else 0,vmax=heat.max() if title!='Cooling difference' else cooling_c)
    map_axes(ax,title); fig.colorbar(m,ax=ax,label='Synthetic degrees C')
plt.tight_layout(); plt.show()
export_json('04_heat.json',dict(synthetic=True,baseline_c=before_mean,scenario_c=after_mean,cooling_c=cooling_c,radius_m=radius_m))


## Teaching lab: Explain raster evidence
**Before using AI:** write a prediction, run the numerical baseline, and explain one surprising result.

**Astra evidence:** send only 04_heat.json and the displayed maps. Use a fresh conversation and record the exact model,
date, interface, tools and prompt. A task with a supplied result tests interpretation, not independent calculation.

**Copyable Astra prompt:**
> Compute the population-weighted cooling from the exported baseline and scenario. Explain why this need not equal an area-average change.

**Copyable Codex task:**
> Calculate both area-average and population-weighted cooling; expose the sampling method and units in the export. Edit the authoring source in `scripts/make_notebooks.py` (or
> `scripts/teaching_labs.py` for labs 11–13), regenerate the notebooks and preserve browser compatibility.
> Use NumPy, Matplotlib and the standard library. Include a small reference case that can be checked by hand.

**Acceptance evidence:** Zero cooling intensity produces zero change; weighting uses all 1200 residents; raster rows correspond to northing.

**Share back:** save your original prediction, changed parameter, result, model response and one limitation.
Download the edited notebook and exports. If you have no model access, exchange explanations with a partner
and evaluate them with the same criteria; do not label that activity as an Astra trial.


## Extend with Codex or Astra
Add a small licensed Landsat/Sentinel-derived raster chip, cloud masks and nodata tests. Do not equate land-surface temperature with indoor exposure or air temperature.

Keep the baseline seed and tests fixed while changing one assumption. Record the model name,
date, exact prompt, response and error metrics. Generated code must pass the same checks.
Download files in `exports/` and your edited notebook; browser storage does not commit changes to GitHub.
See [the project guide](https://github.com/jltobias/JupyterLite-GPT-6-Astra-Geospatial/blob/main/docs/GUIDE.md)
for capabilities, evidence, sources and the route to real data.
